In [ ]:
"""
This train/eval notebook contains the previous iteration of the
CORAL experiment in which target data was adapted to source
characteristics (target to source) and source-fitted classifiers
were used for evaluation.
"""
import pandas as pd
import numpy as np
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, recall_score
import joblib
import os

In [16]:
# Load datasets and pretrained artifacts
source_train_data_path = os.path.join('data', 'processed', 'source', 'train.csv')
source_test_data_path = os.path.join('data', 'processed', 'source', 'test.csv')
target_train_data_path = os.path.join('data', 'processed', 'target', 'train.csv')
target_test_data_path = os.path.join('data', 'processed', 'target', 'test.csv')

label_encoder_path = os.path.join('models', 'label_encoder.joblib')
coral_source_stats_path = os.path.join('models', 'coral_source_stats.joblib')
coral_target_stats_path = os.path.join('models', 'coral_target_stats.joblib')

source_train_df = pd.read_csv(source_train_data_path)
source_test_df = pd.read_csv(source_test_data_path)
target_train_df = pd.read_csv(target_train_data_path)
target_test_df = pd.read_csv(target_test_data_path)

label_encoder = joblib.load(label_encoder_path)
coral_source_stats = joblib.load(coral_source_stats_path)
coral_target_stats = joblib.load(coral_target_stats_path)

print(f"Source train shape: {source_train_df.shape}")
print(f"Source test shape: {source_test_df.shape}")
print(f"Target train shape: {target_train_df.shape}")
print(f"Target test shape: {target_test_df.shape}")
print("Loaded artifacts: label_encoder, coral_source_stats, coral_target_stats")

Source train shape: (1072115, 78)
Source test shape: (268029, 78)
Target train shape: (1856679, 78)
Target test shape: (464170, 78)
Loaded artifacts: label_encoder, coral_source_stats, coral_target_stats


In [17]:
# Sanity checks
# Verify source and target datasets have equivalent feature and label space.

shared_feature_path = os.path.join('data', 'processed', 'shared_feature_space.json')
shared_label_path = os.path.join('data', 'processed', 'shared_label_space.json')

import json
with open(shared_feature_path, 'r') as f:
    shared_feature_payload = json.load(f)
with open(shared_label_path, 'r') as f:
    shared_label_payload = json.load(f)

# Accept either list format or object payload for schema compatibility with preprocessing notebooks.
if isinstance(shared_feature_payload, dict):
    if 'features' not in shared_feature_payload:
        raise ValueError("Expected key 'features' when shared feature payload is a JSON object.")
    shared_features = list(shared_feature_payload['features'])
elif isinstance(shared_feature_payload, list):
    shared_features = list(shared_feature_payload)
else:
    raise ValueError(f"Unsupported shared feature payload type: {type(shared_feature_payload).__name__}")

if isinstance(shared_label_payload, dict):
    if 'labels' not in shared_label_payload:
        raise ValueError("Expected key 'labels' when shared label payload is a JSON object.")
    shared_labels = list(shared_label_payload['labels'])
elif isinstance(shared_label_payload, list):
    shared_labels = list(shared_label_payload)
else:
    raise ValueError(f"Unsupported shared label payload type: {type(shared_label_payload).__name__}")

assert set(shared_features) <= set(source_train_df.columns), "Source train missing shared features!"
assert set(shared_features) <= set(source_test_df.columns), "Source test missing shared features!"
assert set(shared_features) <= set(target_train_df.columns), "Target train missing shared features!"
assert set(shared_features) <= set(target_test_df.columns), "Target test missing shared features!"

# Normalize each split's labels into class-name space before set comparison.
def to_label_name_set(label_series, fitted_label_encoder):
    labels = label_series.dropna()
    if pd.api.types.is_numeric_dtype(labels):
        return set(fitted_label_encoder.inverse_transform(labels.astype(int).to_numpy()))
    return set(labels.astype(str).to_numpy())

expected_label_set = set(map(str, shared_labels))
source_train_label_set = to_label_name_set(source_train_df['Label'], label_encoder)
source_test_label_set = to_label_name_set(source_test_df['Label'], label_encoder)
target_train_label_set = to_label_name_set(target_train_df['Label'], label_encoder)
target_test_label_set = to_label_name_set(target_test_df['Label'], label_encoder)

source_combined_label_set = source_train_label_set | source_test_label_set
target_combined_label_set = target_train_label_set | target_test_label_set

assert source_combined_label_set == expected_label_set, (
    "Combined source label space mismatch vs shared labels! "
    f"Missing={sorted(expected_label_set - source_combined_label_set)}, "
    f"Extra={sorted(source_combined_label_set - expected_label_set)}"
 )
assert target_combined_label_set == expected_label_set, (
    "Combined target label space mismatch vs shared labels! "
    f"Missing={sorted(expected_label_set - target_combined_label_set)}, "
    f"Extra={sorted(target_combined_label_set - expected_label_set)}"
 )
assert source_combined_label_set == target_combined_label_set, (
    "Combined source/target label spaces do not match!"
 )

print("Sanity checks passed: Shared feature space verified for source train/test and target train/test.")
print("Sanity checks passed: Combined source and target label spaces match shared labels.")

Sanity checks passed: Shared feature space verified for source train/test and target train/test.
Sanity checks passed: Combined source and target label spaces match shared labels.


In [ ]:
### Fit Random Forest on source dataset ###

# hyperparameters: n_estimators=100, max_depth=None, min_samples_split=2, min_samples_leaf=1
from sklearn.ensemble import RandomForestClassifier

X_source_train = source_train_df[shared_features]
y_source_train = source_train_df['Label'].astype(int)

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1,
 )
rf.fit(X_source_train, y_source_train)

rf_model_path = os.path.join('models', 'rf_model.joblib')
joblib.dump(rf, rf_model_path)

print('Random Forest trained on source train split.')
print(f'Saved Random Forest model to: {rf_model_path}')

Random Forest trained on source train split.
Saved Random Forest model to: models/rf_model.joblib


In [19]:
### Fit SVM on source dataset ###

# hyperparameters: kernel=linear, C=1.5
X_source_train = source_train_df[shared_features]
y_source_train = source_train_df['Label'].astype(int)

svm = LinearSVC(
    C=1.5,
    tol=1e-2,
    dual=False,
    max_iter=2000,
    random_state=42,
 )
svm.fit(X_source_train, y_source_train)

svm_model_path = os.path.join('models', 'svm_model.joblib')
joblib.dump(svm, svm_model_path)

print('Linear SVM trained on source train split.')
print(f'Saved Linear SVM model to: {svm_model_path}')

Linear SVM trained on source train split.
Saved Linear SVM model to: models/svm_model.joblib


/Users/amazlumyan/Desktop/GitHub/domain-adaptation-poc/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [20]:
### Fit Multilayer Perceptron on source dataset ###

# hyperparameters: hidden_layers_sizes=(64,), alpha=0.001
from sklearn.neural_network import MLPClassifier

X_source_train = source_train_df[shared_features]
y_source_train = source_train_df['Label'].astype(int)

mlp = MLPClassifier(
    hidden_layer_sizes=(64,),
    alpha=0.001,
    random_state=42,
    max_iter=200,
 )
mlp.fit(X_source_train, y_source_train)

mlp_model_path = os.path.join('models', 'mlp_model.joblib')
joblib.dump(mlp, mlp_model_path)

print('MLP trained on source train split.')
print(f'Saved MLP model to: {mlp_model_path}')

MLP trained on source train split.
Saved MLP model to: models/mlp_model.joblib


In [21]:
# ### Preliminary testing classifier (SGD linear SVM) ###

# from sklearn.linear_model import SGDClassifier

# X_source_train = source_train_df[shared_features]
# y_source_train = source_train_df['Label'].astype(int)

# # # terminated after 45min
# # svm = LinearSVC(C=1.0, dual=False, tol=1e-2, max_iter=2000, random_state=42)
# # svm.fit(X_source_train, y_source_train)

# # # terminated after 79min
# # svm = LinearSVC(C=1.0, dual=False, tol=1e-2, max_iter=2000, class_weight='balanced', random_state=42)
# # svm.fit(X_source_train, y_source_train)

# svm = SGDClassifier(
#     loss="hinge",              # linear SVM-style objective
#     alpha=1e-4,                # regularization strength (higher = more regularization)
#     penalty="l2",
#     max_iter=20,               # epochs over data
#     tol=1e-3,                  # early stopping tolerance
#     early_stopping=True,       # holds out validation split and stops when no gain
#     validation_fraction=0.05,
#     n_iter_no_change=3,
#     class_weight="balanced",   # useful for IDS imbalance
#     learning_rate="optimal",
#     average=True,              # often improves stability/generalization
#     random_state=42
# )

# svm.fit(X_source_train, y_source_train)
# print("Fast linear classifier (SGD hinge) trained.")

# print("Linear SVM trained on source train.csv dataset using numeric labels from processed data.")

In [22]:
### Evaluate models ###
# (1) Evaluate all trained models on source test split

source_test_path = os.path.join('data', 'processed', 'source', 'test.csv')
source_test_df = pd.read_csv(source_test_path)

X_source_test = source_test_df[shared_features]
y_source_test = source_test_df['Label'].astype(int)

model_registry = {
    'Linear SVM': svm,
    'Random Forest': rf,
    'MLP': mlp,
}

print('==============================================')
print('SOURCE TEST PERFORMANCE (ALL MODELS)')
print('==============================================')

for model_name, model in model_registry.items():
    y_pred = model.predict(X_source_test)
    accuracy = accuracy_score(y_source_test, y_pred)
    macro_recall = recall_score(y_source_test, y_pred, average='macro', zero_division=0)

    print()
    print(f'[{model_name}]')
    print(f'Accuracy: {accuracy:.6f}')
    print(f'Macro Recall: {macro_recall:.6f}')
    print('Classification Report:')
    print(
        classification_report(
            y_source_test,
            y_pred,
            target_names=label_encoder.classes_,
            zero_division=0,
        )
    )

SOURCE TEST PERFORMANCE (ALL MODELS)

[Linear SVM]
Accuracy: 0.971279
Macro Recall: 0.457559
Classification Report:
                            precision    recall  f1-score   support

                    Benign       0.98      0.99      0.98    201022
                       Bot       0.62      0.01      0.03       390
                      DDoS       0.94      1.00      0.97     25603
             DoS GoldenEye       0.95      0.84      0.89      2057
                  DoS Hulk       0.97      0.94      0.95     34569
          DoS Slowhttptest       0.90      0.79      0.85      1046
             DoS slowloris       0.96      0.58      0.72      1077
               FTP-Patator       0.98      0.65      0.78      1186
              Infiltration       1.00      0.14      0.25         7
               SSH-Patator       0.83      0.01      0.02       644
  Web Attack - Brute Force       0.00      0.00      0.00       294
Web Attack - Sql Injection       0.00      0.00      0.00         4

In [23]:
### Evaluate models ###
# (2) Evaluate all trained models on target test split WITHOUT CORAL domain adaptation

X_target = target_test_df[shared_features]
y_target = target_test_df['Label'].astype(int)

if 'model_registry' not in globals():
    model_registry = {
        'Linear SVM': svm,
        'Random Forest': rf,
        'MLP': mlp,
    }

print('==============================================')
print('TARGET TEST PERFORMANCE (NO CORAL, ALL MODELS)')
print('==============================================')

for model_name, model in model_registry.items():
    y_pred = model.predict(X_target)
    accuracy = accuracy_score(y_target, y_pred)
    macro_recall = recall_score(y_target, y_pred, average='macro', zero_division=0)

    print()
    print(f'[{model_name}]')
    print(f'Accuracy: {accuracy:.6f}')
    print(f'Macro Recall: {macro_recall:.6f}')
    print('Classification Report:')
    print(
        classification_report(
            y_target,
            y_pred,
            target_names=label_encoder.classes_,
            zero_division=0,
        )
    )

TARGET TEST PERFORMANCE (NO CORAL, ALL MODELS)

[Linear SVM]
Accuracy: 0.724825
Macro Recall: 0.137056
Classification Report:
                            precision    recall  f1-score   support

                    Benign       0.79      0.97      0.87    343456
                       Bot       0.00      0.00      0.00     10228
                      DDoS       0.00      0.00      0.00     55561
             DoS GoldenEye       0.06      0.11      0.07      8281
                  DoS Hulk       0.12      0.01      0.02     16733
          DoS Slowhttptest       0.00      0.00      0.00        11
             DoS slowloris       0.26      0.69      0.37      1982
               FTP-Patator       0.00      0.00      0.00        10
              Infiltration       0.00      0.00      0.00     17705
               SSH-Patator       0.00      0.00      0.00     10029
  Web Attack - Brute Force       0.00      0.00      0.00       111
Web Attack - Sql Injection       0.00      0.00      0.00

In [24]:
# ### Evaluate models ###
# # (3) Evaluate all trained models on target test split WITH CORAL domain adaptation
# # Improvements implemented:
# # - increase inverse-square-root spectral floor (eps sweep)
# # - add CORAL damping via convex blend

# def stable_symmetric_matrix_power(matrix, power, eps=1e-6):
#     # Enforce float64 + symmetry before eigendecomposition to reduce numerical drift.
#     matrix = np.asarray(matrix, dtype=np.float64)
#     matrix = (matrix + matrix.T) / 2.0

#     eigenvalues, eigenvectors = np.linalg.eigh(matrix)
#     # Add minimum ridge and clip eigenvalues so inverse/sqrt powers stay finite.
#     ridge = max(eps, float(-eigenvalues.min() + eps)) if eigenvalues.min() <= 0 else eps
#     clipped_eigenvalues = np.clip(eigenvalues + ridge, eps, None)

#     powered = eigenvectors @ np.diag(clipped_eigenvalues ** power) @ eigenvectors.T
#     return (powered + powered.T) / 2.0, float(eigenvalues.min()), ridge

# if 'model_registry' not in globals():
#     model_registry = {
#         'Linear SVM': svm,
#         'Random Forest': rf,
#         'MLP': mlp,
#     }

# # Validate and load source/target CORAL statistics.
# # These values are the new artifacts exported from preprocessing and power the
# # improved adaptation path (schema checks + diagnostics + shrinkage metadata).
# source_feature_order = coral_source_stats.get('feature_order')
# target_feature_order = coral_target_stats.get('feature_order')

# if source_feature_order is None or target_feature_order is None:
#     raise KeyError("CORAL stats must include 'feature_order' metadata.")
# if list(source_feature_order) != list(target_feature_order):
#     raise ValueError("Source and target CORAL stats feature_order do not match.")
# if list(shared_features) != list(source_feature_order):
#     raise ValueError(
#         "Runtime shared feature order does not match CORAL stats feature_order. "
#         "Regenerate stats or align feature ordering before evaluation."
#     )

# # Extract source/target statistics used by CORAL transform construction.
# source_mean = np.asarray(coral_source_stats['mean'], dtype=np.float64)
# source_cov = np.asarray(coral_source_stats['covariance'], dtype=np.float64)

# target_mean = np.asarray(coral_target_stats['mean'], dtype=np.float64)
# target_cov = np.asarray(coral_target_stats['covariance'], dtype=np.float64)

# n_features = len(source_feature_order)
# if source_mean.shape[0] != n_features or target_mean.shape[0] != n_features:
#     raise ValueError("CORAL mean vector length does not match feature_order length.")
# if source_cov.shape != (n_features, n_features) or target_cov.shape != (n_features, n_features):
#     raise ValueError("CORAL covariance shape does not match feature_order length.")

# # Convert target features to numpy using validated canonical feature order.
# X_target_np = X_target[source_feature_order].to_numpy(dtype=np.float64)

# # Center target data using target mean before CORAL transform.
# X_target_centered = X_target_np - target_mean

# # Pull covariance diagnostics saved during preprocessing for traceability in output.
# source_estimator = coral_source_stats.get('covariance_estimator', 'unspecified')
# target_estimator = coral_target_stats.get('covariance_estimator', 'unspecified')
# source_shrinkage = coral_source_stats.get('covariance_shrinkage')
# target_shrinkage = coral_target_stats.get('covariance_shrinkage')
# source_cond = coral_source_stats.get('covariance_condition_number')
# target_cond = coral_target_stats.get('covariance_condition_number')
# source_max_eig = coral_source_stats.get('max_eigenvalue')
# target_max_eig = coral_target_stats.get('max_eigenvalue')
# source_min_eig_saved = coral_source_stats.get('min_eigenvalue_before_regularization')
# target_min_eig_saved = coral_target_stats.get('min_eigenvalue_before_regularization')

# # Improvement #2: spectral-floor config grid for stable inverse/sqrt covariance powers.
# spectral_floors = [1e-4, 1e-3]

# # Improvement #3: damping config grid for partial CORAL adaptation strength.
# coral_lambdas = [0.1, 0.25, 0.5, 0.75, 1.0]

# # Track all config evaluations and keep the best-performing adapted target variant per model.
# sweep_rows = []
# best_results_by_model = {model_name: None for model_name in model_registry}

# for eps in spectral_floors:
#     # Build one CORAL transform for this eps: A = Ct^(-1/2) * Cs^(1/2).
#     target_cov_inv_sqrt, target_min_eig, target_ridge = stable_symmetric_matrix_power(target_cov, -0.5, eps=eps)
#     source_cov_sqrt, source_min_eig, source_ridge = stable_symmetric_matrix_power(source_cov, 0.5, eps=eps)

#     A_coral_candidate = target_cov_inv_sqrt @ source_cov_sqrt

#     # Produce the full-CORAL target-feature variant for this eps.
#     X_target_coral_full = (X_target_centered @ A_coral_candidate) + source_mean

#     if not np.isfinite(X_target_coral_full).all():
#         raise ValueError(f"CORAL transform produced non-finite values for eps={eps}.")

#     for coral_lambda in coral_lambdas:
#         # Produce a damped target-feature variant for this (eps, lambda) config.
#         X_target_adapted = X_target_np + coral_lambda * (X_target_coral_full - X_target_np)

#         if not np.isfinite(X_target_adapted).all():
#             raise ValueError(
#                 f"Damped CORAL produced non-finite values for eps={eps}, lambda={coral_lambda}."
#             )

#         X_target_adapted_df = pd.DataFrame(
#             X_target_adapted,
#             columns=source_feature_order,
#             index=X_target.index,
#         )

#         # Score all trained models on this adapted target-test variant.
#         for model_name, model in model_registry.items():
#             y_pred = model.predict(X_target_adapted_df)
#             accuracy = accuracy_score(y_target, y_pred)
#             macro_recall = recall_score(y_target, y_pred, average='macro', zero_division=0)

#             row = {
#                 'model': model_name,
#                 'eps': eps,
#                 'lambda': coral_lambda,
#                 'macro_recall': macro_recall,
#                 'accuracy': accuracy,
#                 'source_min_eig_runtime': source_min_eig,
#                 'target_min_eig_runtime': target_min_eig,
#                 'source_ridge_runtime': source_ridge,
#                 'target_ridge_runtime': target_ridge,
#             }
#             sweep_rows.append(row)

#             current_best = best_results_by_model[model_name]
#             is_better = (
#                 current_best is None
#                 or macro_recall > current_best['macro_recall']
#                 or (
#                     macro_recall == current_best['macro_recall']
#                     and accuracy > current_best['accuracy']
#                 )
#             )

#             if is_better:
#                 best_results_by_model[model_name] = {
#                     **row,
#                     'A_coral': A_coral_candidate,
#                     'X_target_coral_df': X_target_adapted_df,
#                     'y_pred': y_pred,
#                 }

# if not sweep_rows:
#     raise RuntimeError("No CORAL sweep results were produced.")

# # Sort all configs to make top-performing settings easy to inspect.
# results_df = pd.DataFrame(sweep_rows).sort_values(
#     ['model', 'macro_recall', 'accuracy', 'eps', 'lambda'],
#     ascending=[True, False, False, True, True]
# )

# # Keep backwards-compatible variables for downstream analysis using SVM best config.
# best_result = best_results_by_model['Linear SVM']
# A_coral = best_result['A_coral']
# X_target_coral_df = best_result['X_target_coral_df']
# y_target_coral_pred = best_result['y_pred']

# print(f"Source covariance estimator: {source_estimator}, shrinkage={source_shrinkage}")
# print(f"Target covariance estimator: {target_estimator}, shrinkage={target_shrinkage}")
# print(f"Source covariance eig(min/max saved): {source_min_eig_saved}/{source_max_eig}")
# print(f"Target covariance eig(min/max saved): {target_min_eig_saved}/{target_max_eig}")
# print(f"Source covariance condition number (saved): {source_cond}")
# print(f"Target covariance condition number (saved): {target_cond}")
# print()
# print('==============================================')
# print('TARGET TEST PERFORMANCE (WITH CORAL, ALL MODELS)')
# print('==============================================')

# for model_name in model_registry:
#     model_rows = results_df[results_df['model'] == model_name]
#     selected = best_results_by_model[model_name]

#     print()
#     print(f'[{model_name}] CORAL sweep top 5 configs:')
#     print(
#         model_rows[['eps', 'lambda', 'macro_recall', 'accuracy']].head(5).to_string(index=False)
#     )

#     print(
#         f"Selected best CORAL config: eps={selected['eps']:.0e}, "
#         f"lambda={selected['lambda']}, "
#         f"macro_recall={selected['macro_recall']:.6f}, "
#         f"accuracy={selected['accuracy']:.6f}"
#     )
#     print(
#         f"Source min eig/runtime ridge: {selected['source_min_eig_runtime']:.6e} / "
#         f"{selected['source_ridge_runtime']:.6e}"
#     )
#     print(
#         f"Target min eig/runtime ridge: {selected['target_min_eig_runtime']:.6e} / "
#         f"{selected['target_ridge_runtime']:.6e}"
#     )
#     print(f'\n[{model_name}]')
#     print(f"Accuracy: {selected['accuracy']:.6f}")
#     print(f"Macro Recall: {selected['macro_recall']:.6f}")
#     print('Classification Report:')
#     print(
#         classification_report(
#             y_target,
#             selected['y_pred'],
#             target_names=label_encoder.classes_,
#             zero_division=0,
#         )
#     )

In [25]:
### Evaluate models ###
# (3) Evaluate all trained models on target test split WITH CORAL domain adaptation
# Note: no CORAL sweeps / parameter tuning via target-label feedback

from sklearn.metrics import recall_score

def stable_symmetric_matrix_power(matrix, power, eps=1e-6):
    # Enforce float64 + symmetry before eigendecomposition to reduce numerical drift.
    matrix = np.asarray(matrix, dtype=np.float64)
    matrix = (matrix + matrix.T) / 2.0

    eigenvalues, eigenvectors = np.linalg.eigh(matrix)
    # Add minimum ridge and clip eigenvalues so inverse/sqrt powers stay finite.
    ridge = max(eps, float(-eigenvalues.min() + eps)) if eigenvalues.min() <= 0 else eps
    clipped_eigenvalues = np.clip(eigenvalues + ridge, eps, None)

    powered = eigenvectors @ np.diag(clipped_eigenvalues ** power) @ eigenvectors.T
    return (powered + powered.T) / 2.0, float(eigenvalues.min()), ridge

if 'model_registry' not in globals():
    model_registry = {
        'Linear SVM': svm,
        'Random Forest': rf,
        'MLP': mlp,
    }

# Validate and load source/target CORAL statistics.
source_feature_order = coral_source_stats.get('feature_order')
target_feature_order = coral_target_stats.get('feature_order')

if source_feature_order is None or target_feature_order is None:
    raise KeyError("CORAL stats must include 'feature_order' metadata.")
if list(source_feature_order) != list(target_feature_order):
    raise ValueError("Source and target CORAL stats feature_order do not match.")
if list(shared_features) != list(source_feature_order):
    raise ValueError(
        "Runtime shared feature order does not match CORAL stats feature_order. "
        "Regenerate stats or align feature ordering before evaluation."
    )

# Extract source/target statistics used by CORAL transform construction.
source_mean = np.asarray(coral_source_stats['mean'], dtype=np.float64)
source_cov = np.asarray(coral_source_stats['covariance'], dtype=np.float64)

target_mean = np.asarray(coral_target_stats['mean'], dtype=np.float64)
target_cov = np.asarray(coral_target_stats['covariance'], dtype=np.float64)

n_features = len(source_feature_order)
if source_mean.shape[0] != n_features or target_mean.shape[0] != n_features:
    raise ValueError("CORAL mean vector length does not match feature_order length.")
if source_cov.shape != (n_features, n_features) or target_cov.shape != (n_features, n_features):
    raise ValueError("CORAL covariance shape does not match feature_order length.")

# Convert target features to numpy using validated canonical feature order.
X_target_np = X_target[source_feature_order].to_numpy(dtype=np.float64)

# Center target data using target mean before CORAL transform.
X_target_centered = X_target_np - target_mean

# Pull covariance diagnostics saved during preprocessing for traceability in output.
source_estimator = coral_source_stats.get('covariance_estimator', 'unspecified')
target_estimator = coral_target_stats.get('covariance_estimator', 'unspecified')
source_shrinkage = coral_source_stats.get('covariance_shrinkage')
target_shrinkage = coral_target_stats.get('covariance_shrinkage')
source_cond = coral_source_stats.get('covariance_condition_number')
target_cond = coral_target_stats.get('covariance_condition_number')
source_max_eig = coral_source_stats.get('max_eigenvalue')
target_max_eig = coral_target_stats.get('max_eigenvalue')
source_min_eig_saved = coral_source_stats.get('min_eigenvalue_before_regularization')
target_min_eig_saved = coral_target_stats.get('min_eigenvalue_before_regularization')

# Standard CORAL transform with a single default regularization setting.
target_cov_inv_sqrt, target_min_eig, target_ridge = stable_symmetric_matrix_power(target_cov, -0.5)
source_cov_sqrt, source_min_eig, source_ridge = stable_symmetric_matrix_power(source_cov, 0.5)

A_coral = target_cov_inv_sqrt @ source_cov_sqrt
X_target_coral_np = (X_target_centered @ A_coral) + source_mean

if not np.isfinite(X_target_coral_np).all():
    raise ValueError("CORAL transform produced non-finite values.")

X_target_coral_df = pd.DataFrame(
    X_target_coral_np,
    columns=source_feature_order,
    index=X_target.index,
)

print(f"Source covariance estimator: {source_estimator}, shrinkage={source_shrinkage}")
print(f"Target covariance estimator: {target_estimator}, shrinkage={target_shrinkage}")
print(f"Source covariance eig(min/max saved): {source_min_eig_saved}/{source_max_eig}")
print(f"Target covariance eig(min/max saved): {target_min_eig_saved}/{target_max_eig}")
print(f"Source covariance condition number (saved): {source_cond}")
print(f"Target covariance condition number (saved): {target_cond}")
print(f"Source min eig/runtime ridge: {source_min_eig:.6e} / {source_ridge:.6e}")
print(f"Target min eig/runtime ridge: {target_min_eig:.6e} / {target_ridge:.6e}")
print()
print('==============================================')
print('TARGET TEST PERFORMANCE (WITH CORAL, ALL MODELS)')
print('==============================================')

for model_name, model in model_registry.items():
    y_pred = model.predict(X_target_coral_df)
    accuracy = accuracy_score(y_target, y_pred)
    macro_recall = recall_score(y_target, y_pred, average='macro', zero_division=0)

    print()
    print(f'[{model_name}]')
    print(f"Accuracy: {accuracy:.6f}")
    print(f"Macro Recall: {macro_recall:.6f}")
    print('Classification Report:')
    print(
        classification_report(
            y_target,
            y_pred,
            target_names=label_encoder.classes_,
            zero_division=0,
        )
    )

Source covariance estimator: LedoitWolf, shrinkage=0.020618143921211536
Target covariance estimator: LedoitWolf, shrinkage=1.0
Source covariance eig(min/max saved): 0.018475999098227745/14.94379627734913
Target covariance eig(min/max saved): 552.9764361083896/552.9764361083896
Source covariance condition number (saved): 808.8220938905803
Target covariance condition number (saved): 1.0
Source min eig/runtime ridge: 1.847600e-02 / 1.000000e-06
Target min eig/runtime ridge: 5.529764e+02 / 1.000000e-06

TARGET TEST PERFORMANCE (WITH CORAL, ALL MODELS)

[Linear SVM]
Accuracy: 0.739934
Macro Recall: 0.076923
Classification Report:
                            precision    recall  f1-score   support

                    Benign       0.74      1.00      0.85    343456
                       Bot       0.00      0.00      0.00     10228
                      DDoS       0.00      0.00      0.00     55561
             DoS GoldenEye       0.00      0.00      0.00      8281
                  DoS Hulk